# Day 7 - Databricks
## RAG over Internal Documents

Databricks is a leading data and AI platform, built around the lakehouse architecture that combines the flexibility of a data lake with the performance and governance of a data warehouse. It is where many data engineering and ML teams already store, process and serve their data.

Databricks supports the `VECTOR` data type and `VECTOR_COSINE_SIMILARITY` function natively in Databricks SQL, making it straightforward to add semantic search to data that already lives in the lakehouse, without introducing a separate vector database.

**When would you reach for this?**
- Your data and ML workflows already live in Databricks
- You want RAG over documents stored in the lakehouse
- Your team works in SQL and PySpark and prefers to stay there

**The use case:** A retrieval-augmented generation (RAG) system over internal technical documents - architecture notes, runbooks, API guides and onboarding materials. Users ask questions in natural language and the system finds the most relevant documents to ground a response.

## 1. Setup

### Prerequisites

- A Databricks community edition account with access to a SQL warehouse
- Ollama running locally with the `all-minilm` model pulled
- Python 3.12 with a virtual environment

### Generate a Personal Access Token

1. In the Databricks UI, click your username in the top bar and select **Settings**
2. Click **Developer**
3. Next to **Access tokens**, click **Manage**
4. Click **Generate new token**, give it a name, select **BI Tools** as the scope type and click **Generate**
5. Copy the token immediately - it is only shown once

### Find Your SQL Warehouse Connection Details

1. In the Databricks UI, go to **SQL > SQL Warehouses**
2. Click your warehouse and select **Connection details**
3. Note down the **Server hostname** and **HTTP path**

### Set Environment Variables

```bash
export DATABRICKS_SERVER_HOSTNAME="dbc-xxxx.cloud.databricks.com"
export DATABRICKS_HTTP_PATH="/sql/1.0/warehouses/xxxx"
export DATABRICKS_TOKEN="your-personal-access-token"
```

### Install Python Dependencies

In [1]:
%pip install ollama==0.6.2 \
             databricks-sql-connector==4.4.0 \
             pandas==3.0.3 \
             tqdm==4.67.1 --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import ollama
import os
import pandas as pd
import random

from databricks import sql
from tqdm.notebook import tqdm

### Configuration

In [3]:
DATABRICKS_SERVER_HOSTNAME = os.environ["DATABRICKS_SERVER_HOSTNAME"]
DATABRICKS_HTTP_PATH       = os.environ["DATABRICKS_HTTP_PATH"]
DATABRICKS_TOKEN           = os.environ["DATABRICKS_TOKEN"]
CATALOG_NAME               = "workspace"
SCHEMA_NAME                = "wiki_docs"
TABLE_NAME                 = "documents"
LLM_EMBEDDING              = "all-minilm"
NUM_DOCS                   = 200
RANDOM_SEED                = 42

> **Note:** `NUM_DOCS` controls the size of the generated dataset. 200 is the recommended default. Embedding generation runs locally via Ollama and is single-threaded. At 200 documents the notebook runs comfortably. Larger values will work but will take proportionally longer.

### Verify Ollama is Running

In [4]:
ollama_ready = False

try:
    models = ollama.list()
    model_names = [m.model for m in models.models]
    assert any(LLM_EMBEDDING in m for m in model_names)
    print(f"Model '{LLM_EMBEDDING}' is ready.")
    ollama_ready = True
except ConnectionError:
    print("ERROR: Ollama is not running. Start it with: ollama serve")
except AssertionError:
    print(f"ERROR: Model not found. Run: ollama pull {LLM_EMBEDDING}")

Model 'all-minilm' is ready.


In [5]:
assert ollama_ready, "Please fix the Ollama issue above before continuing."

### Determine Embedding Dimensions

In [6]:
def get_embedding(text: str) -> list:
    response = ollama.embeddings(model = LLM_EMBEDDING, prompt = text)
    return response["embedding"]

test_embedding = get_embedding("internal technical documentation")
EMBEDDING_DIMS = len(test_embedding)
print(f"Embedding dimensions: {EMBEDDING_DIMS}")

Embedding dimensions: 384


### Connect to Databricks

In [7]:
conn = sql.connect(
    server_hostname = DATABRICKS_SERVER_HOSTNAME,
    http_path       = DATABRICKS_HTTP_PATH,
    access_token    = DATABRICKS_TOKEN
)

cursor = conn.cursor()

print("Connected to Databricks.")

Connected to Databricks.


### Create Schema

In [8]:
cursor.execute(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")
cursor.execute(f"USE {CATALOG_NAME}.{SCHEMA_NAME}")

print(f"Using schema: {CATALOG_NAME}.{SCHEMA_NAME}")

Using schema: workspace.wiki_docs


## 2. The Dataset

We generate synthetic internal technical wiki documents from pools of categories, authors and content templates. Each document has a title, category, author and a prose content field - the content is what we embed and search over.

The categories represent typical internal documentation types: Architecture, Runbooks, API, Security, Onboarding, Data Engineering, ML Platform and Incident Reports.

In [9]:
random.seed(RANDOM_SEED)

CATEGORIES = [
    "Architecture",
    "Runbooks",
    "API",
    "Security",
    "Onboarding",
    "Data Engineering",
    "ML Platform",
    "Incident Reports",
]

FICTIONAL_AUTHORS = [
    "Jordan Blake", "Morgan Chen", "Taylor Osei", "Riley Kumar",
    "Casey Petrov", "Avery Singh", "Quinn Nakamura", "Drew Andersen",
    "Skyler Mbeki", "Reese Fontaine",
]

CONTENT_TEMPLATES = {
    "Architecture": [
        "This document describes the high-level architecture of the {system} platform. The system is built on a microservices model with {pattern} as the primary communication pattern. Services are deployed on {infra} and managed via {tool}. Key design decisions include the use of {decision} to ensure scalability and fault tolerance.",
        "The {system} architecture follows a {style} approach, separating concerns across {layers} layers. Data flows from ingestion through transformation to serving via a {pipeline} pipeline. Infrastructure is provisioned using {tool} and monitored with {monitoring}.",
        "This architecture document covers the design of the {system} service. The service handles {volume} requests per day and is designed for {goal}. It uses {pattern} for asynchronous processing and {storage} for persistent state.",
    ],
    "Runbooks": [
        "This runbook covers the procedure for {task} in the {system} environment. Follow these steps in order during an incident. First, verify the {check} is responding correctly. If not, escalate to the {team} team and open a P{priority} incident ticket. Roll back using {rollback} if the issue persists after {timeout} minutes.",
        "Use this runbook when {scenario} occurs in production. The on-call engineer should first check {dashboard} for anomalies. Run the {script} script to gather diagnostics. If {condition}, restart the {service} service and monitor for {window} minutes before closing the incident.",
        "This runbook documents the steps to {task} for the {system} service. Prerequisites: access to {access}. Steps: (1) Verify {check}. (2) Run {command}. (3) Confirm {confirmation}. Contact {team} if any step fails unexpectedly.",
    ],
    "API": [
        "The {system} API provides endpoints for {purpose}. Authentication uses {auth}. All requests must include the {header} header. Rate limits are {rate} requests per minute per client. The base URL is {url}. Errors follow the {format} format with standard HTTP status codes.",
        "This document covers the {system} REST API. The API supports {methods} methods and returns {format} responses. Pagination uses {pagination}. Webhooks are available for {events} events. The API version is {version} and is stable until {date}.",
        "The {system} API enables integration with {purpose}. Use the {sdk} SDK for Python or JavaScript clients. Authentication requires a {auth} token passed in the Authorization header. The sandbox environment is available at {url} for testing.",
    ],
    "Security": [
        "This document outlines the security requirements for {system}. All data in transit must use {encryption}. Access is controlled via {access_control}. Audit logs are retained for {retention} days. Security reviews are required before {trigger}. Contact the {team} team for exceptions.",
        "The security policy for {system} requires {requirement}. Credentials must be rotated every {rotation} days. Multi-factor authentication is mandatory for {access}. Penetration testing is conducted {frequency}. Vulnerabilities must be patched within {sla} days of discovery.",
        "This document covers the data classification and handling requirements for {system}. Data classified as {classification} must be encrypted at rest using {encryption}. Access is restricted to {roles} roles. Data must not be stored in {prohibited} environments.",
    ],
    "Onboarding": [
        "Welcome to the {team} team. This guide covers everything you need to get set up in your first {period}. You will need access to {tools}. Your onboarding buddy is assigned on day one. Complete the {training} training modules in the first week. Your first task will be {task}.",
        "This onboarding document is for new {role} engineers joining the {team} team. Start by requesting access to {systems}. Set up your local development environment following the {guide} guide. Attend the {meeting} meeting on your first day. Review the {docs} documentation before your first sprint.",
        "New joiners to the {team} team should complete this checklist in the first {period}. Install {tools} on your machine. Request access to {systems} via the IT portal. Read the {docs} documentation. Complete the {training} certification. Schedule a 1:1 with your manager in the first week.",
    ],
    "Data Engineering": [
        "This document describes the {pipeline} pipeline for the {system} data product. Data is ingested from {source} every {frequency}. Transformations are applied using {tool} and written to {destination} in {format} format. SLA is {sla} hours from source availability. Failures trigger alerts to {team}.",
        "The {system} data pipeline runs on {schedule} and processes {volume} records per run. It reads from {source}, applies {transformation} transformations and writes to {destination}. The pipeline is orchestrated by {orchestrator} and monitored via {monitoring}. On-call rotation handles incidents.",
        "This guide covers the data model for {system}. The primary tables are {tables}. Data is partitioned by {partition} for query performance. Retention policy is {retention} days for raw data and {processed_retention} days for processed data. Schema changes require a migration plan approved by {team}.",
    ],
    "ML Platform": [
        "The {system} ML platform supports the full model lifecycle from {stage1} to {stage2}. Models are trained using {framework} on {compute}. Experiments are tracked in {tracker}. Models are deployed to {serving} for real-time inference or {batch} for batch scoring. Retraining is triggered {trigger}.",
        "This document covers model deployment on the {system} platform. Models must pass {checks} checks before promotion to production. Deployment uses {strategy} strategy for zero-downtime releases. Monitoring tracks {metrics} metrics and triggers retraining when {threshold}. Model artifacts are stored in {registry}.",
        "The {system} feature store provides {features} features for model training and serving. Features are computed {frequency} and stored in {storage}. Online serving latency is under {latency}ms. Feature definitions are managed in {tool}. New features require review by the {team} team.",
    ],
    "Incident Reports": [
        "Incident {id}: {system} experienced {issue} on {date}. Duration: {duration} minutes. Impact: {impact}. Root cause: {cause}. The incident was triggered by {trigger} and detected via {detection}. Mitigation involved {mitigation}. Action items: {actions}.",
        "Post-incident review for {system} outage. The incident began at {time} when {trigger} caused {issue}. On-call engineer {responder} was paged and began investigation. Root cause identified as {cause}. Service was restored after {duration} minutes. Prevention: {prevention}.",
        "This incident report covers the {system} degradation on {date}. Severity: P{severity}. Affected users: {users}. The degradation was caused by {cause} following {preceding_event}. The team resolved the issue by {resolution}. Follow-up tasks have been filed in the backlog.",
    ],
}

TEMPLATE_VARS = {
    "system":            ["DataPlatform", "AnalyticsAPI", "StreamProcessor", "ModelServing", "DataVault", "EventBus", "SearchEngine", "ReportingService"],
    "pattern":           ["event-driven messaging", "request-response", "publish-subscribe", "streaming"],
    "infra":             ["Kubernetes", "serverless functions", "managed containers", "virtual machines"],
    "tool":              ["Terraform", "Helm", "Ansible", "CloudFormation", "Pulumi"],
    "decision":          ["circuit breakers", "idempotent writes", "distributed tracing", "blue-green deployments"],
    "style":             ["layered", "hexagonal", "event-driven", "CQRS"],
    "layers":            ["three", "four", "five"],
    "pipeline":          ["batch ETL", "streaming", "lambda architecture", "micro-batch"],
    "monitoring":        ["Grafana", "Datadog", "CloudWatch", "Prometheus"],
    "volume":            ["10 million", "50 million", "500 thousand", "2 million"],
    "goal":              ["high availability", "low latency", "horizontal scalability", "data consistency"],
    "storage":           ["Delta Lake", "PostgreSQL", "object storage", "Redis"],
    "task":              ["restarting the service", "rolling back a deployment", "clearing a stuck queue", "rotating credentials", "scaling the cluster"],
    "check":             ["health endpoint", "metrics dashboard", "log stream", "alerting system"],
    "team":              ["platform", "data engineering", "ML", "security", "infrastructure", "SRE"],
    "priority":          ["1", "2", "3"],
    "rollback":          ["the previous Docker image", "the last known good configuration", "a database snapshot"],
    "timeout":           ["15", "30", "60"],
    "scenario":          ["high error rates", "latency spikes", "database connection exhaustion", "memory pressure"],
    "dashboard":         ["the SLO dashboard", "Grafana", "Datadog", "CloudWatch"],
    "script":            ["diagnose.sh", "health_check.py", "collect_logs.sh"],
    "condition":         ["error rates exceed 5%", "latency exceeds 500ms", "the queue depth exceeds 10,000"],
    "service":           ["API gateway", "worker service", "scheduler", "cache"],
    "window":            ["10", "15", "30"],
    "access":            ["production console", "the secrets manager", "the admin panel", "VPN access"],
    "command":           ["kubectl rollout restart", "systemctl restart", "docker-compose up"],
    "confirmation":      ["health checks pass", "error rates return to baseline", "the queue is draining"],
    "purpose":           ["managing user accounts", "querying analytics data", "triggering batch jobs", "streaming events"],
    "auth":              ["OAuth 2.0", "API key", "JWT", "mTLS"],
    "header":            ["Authorization", "X-API-Key", "X-Request-ID"],
    "rate":              ["100", "500", "1000", "60"],
    "url":               ["https://api.internal.example.com", "https://sandbox.api.example.com"],
    "format":            ["JSON", "JSON:API", "HAL", "Protobuf"],
    "methods":           ["GET, POST, PUT and DELETE", "GET and POST", "GET, POST and PATCH"],
    "pagination":        ["cursor-based pagination", "offset pagination", "keyset pagination"],
    "events":            ["create, update and delete", "status change", "payment and refund"],
    "version":           ["v1", "v2", "v3"],
    "date":              ["Q4 2025", "Q1 2026", "Q2 2026"],
    "sdk":               ["official Python", "community JavaScript", "Go"],
    "encryption":        ["TLS 1.3", "AES-256", "end-to-end encryption"],
    "access_control":    ["role-based access control", "attribute-based access control", "OAuth scopes"],
    "retention":         ["90", "180", "365"],
    "trigger":           ["new service deployments", "schema changes", "quarterly reviews", "drift detection"],
    "requirement":       ["all secrets to be stored in the vault", "MFA for all production access", "encrypted storage for PII"],
    "rotation":          ["30", "60", "90"],
    "frequency":         ["annually", "quarterly", "monthly", "hourly", "daily"],
    "sla":               ["7", "14", "30", "4"],
    "classification":    ["confidential", "internal", "restricted", "public"],
    "prohibited":        ["personal", "unmanaged cloud", "public"],
    "roles":             ["admin", "engineer", "analyst", "read-only"],
    "period":            ["30 days", "two weeks", "first month"],
    "tools":             ["Slack, GitHub and Jira", "VS Code, Docker and kubectl", "Python, dbt and Airflow"],
    "training":          ["security awareness", "data handling", "platform engineering"],
    "role":              ["data engineering", "ML", "platform", "backend"],
    "systems":           ["GitHub, AWS and Databricks", "Jira, Confluence and Datadog", "Snowflake, dbt and Airflow"],
    "guide":             ["local development setup", "engineering handbook", "platform quickstart"],
    "meeting":           ["all-hands", "team standup", "engineering sync"],
    "docs":              ["architecture", "engineering handbook", "platform"],
    "source":            ["the production database", "the event stream", "the data lake", "the CRM system"],
    "destination":       ["Delta Lake", "the data warehouse", "the feature store", "object storage"],
    "orchestrator":      ["Apache Airflow", "Databricks Workflows", "Prefect", "dbt Cloud"],
    "transformation":    ["deduplication and enrichment", "aggregation and filtering", "normalisation and validation"],
    "tables":            ["events, sessions and users", "orders, products and customers", "transactions, accounts and merchants"],
    "partition":         ["date", "region", "event type"],
    "process_retention": ["180", "365", "730"],
    "stage1":            ["experimentation", "data preparation", "feature engineering"],
    "stage2":            ["production serving", "batch inference", "A/B testing"],
    "framework":         ["PyTorch", "scikit-learn", "XGBoost", "TensorFlow"],
    "compute":           ["GPU clusters", "CPU clusters", "serverless compute"],
    "tracker":           ["MLflow", "Weights and Biases", "Neptune"],
    "serving":           ["the model serving endpoint", "a REST API", "a streaming inference service"],
    "batch":             ["the batch scoring pipeline", "a scheduled Databricks job"],
    "checks":            ["unit test, integration test and bias", "performance, latency and fairness"],
    "strategy":          ["canary", "blue-green", "rolling"],
    "metrics":           ["accuracy, latency and drift", "precision, recall and throughput"],
    "threshold":         ["accuracy drops below 95%", "drift exceeds the threshold", "error rate increases"],
    "registry":          ["MLflow Model Registry", "the artifact store", "Unity Catalog"],
    "features":          ["user behaviour", "product interaction", "transaction history"],
    "latency":           ["10", "20", "50", "100"],
    "id":                ["INC-1042", "INC-2317", "INC-0891", "INC-3405"],
    "issue":             ["elevated error rates", "complete service outage", "database connection failures", "latency degradation"],
    "duration":          ["14", "47", "92", "23", "8"],
    "impact":            ["all API requests failed", "read operations degraded by 40%", "batch jobs delayed by 2 hours"],
    "cause":             ["a misconfigured deployment", "a database migration with a missing index", "resource exhaustion", "a dependency timeout"],
    "detection":         ["automated alerting", "customer report", "the on-call engineer"],
    "mitigation":        ["rolling back the deployment", "restarting the affected services", "scaling up the database"],
    "actions":           ["add integration tests, improve runbook and increase alert sensitivity"],
    "time":              ["02:14 UTC", "14:32 UTC", "09:07 UTC"],
    "responder":         ["the on-call engineer", "the platform team lead", "the SRE on duty"],
    "prevention":        ["improved deployment checks", "additional monitoring", "schema change review process"],
    "severity":          ["1", "2", "3"],
    "users":             ["all users", "approximately 30% of users", "enterprise customers only"],
    "preceding_event":   ["a configuration change", "a routine deployment", "a database upgrade"],
    "resolution":        ["rolling back the change", "restarting the affected services", "applying a hotfix"],
    "schedule":          ["hourly", "daily at 02:00 UTC", "every 15 minutes"],
    "schedule2":         ["hourly", "nightly", "every 6 hours"],
}

TITLE_TEMPLATES = {
    "Architecture":      ["{system} Architecture Overview", "{system} System Design", "{system} Technical Architecture"],
    "Runbooks":          ["{system} Runbook: {task}", "{system} Operations Guide", "Runbook: {scenario} in {system}"],
    "API":               ["{system} API Reference", "{system} API Integration Guide", "{system} REST API Documentation"],
    "Security":          ["{system} Security Policy", "{system} Data Classification Guide", "{system} Access Control Policy"],
    "Onboarding":        ["{team} Team Onboarding Guide", "New {role} Engineer Onboarding", "{team} Team Starter Guide"],
    "Data Engineering":  ["{system} Data Pipeline Documentation", "{system} Data Model Guide", "{system} ETL Runbook"],
    "ML Platform":       ["{system} ML Platform Guide", "{system} Model Deployment Documentation", "{system} Feature Store Guide"],
    "Incident Reports":  ["Incident Report: {system} {issue}", "Post-Incident Review: {system}", "{system} Outage Report"],
}

def fill_template(template: str) -> str:
    import re
    placeholders = re.findall(r"\{(\w+)\}", template)
    result = template
    for key in placeholders:
        if key in TEMPLATE_VARS:
            result = result.replace(f"{{{key}}}", random.choice(TEMPLATE_VARS[key]), 1)
    return result

def generate_document(doc_id: int) -> dict:
    category = random.choice(CATEGORIES)
    title    = fill_template(random.choice(TITLE_TEMPLATES[category]))
    content  = fill_template(random.choice(CONTENT_TEMPLATES[category]))
    return {
        "DOC_ID":   f"DOC{doc_id:05d}",
        "TITLE":    title,
        "CATEGORY": category,
        "AUTHOR":   random.choice(FICTIONAL_AUTHORS),
        "CONTENT":  content,
    }

documents = [generate_document(i) for i in range(NUM_DOCS)]
df = pd.DataFrame(documents)

print(f"Generated {len(df)} documents")
print(f"\nCategory distribution:")
print(df["CATEGORY"].value_counts().to_string())
df.head()

Generated 200 documents

Category distribution:
CATEGORY
Runbooks            30
Incident Reports    30
Data Engineering    25
Onboarding          25
Architecture        24
ML Platform         24
Security            22
API                 20


,DOC_ID,TITLE,CATEGORY,AUTHOR,CONTENT
0,DOC00000,DataVault Runbook: rolling back a deployment,Runbooks,Morgan Chen,This runbook covers the procedure for rolling ...
1,DOC00001,DataPlatform Security Policy,Security,Casey Petrov,This document covers the data classification a...
2,DOC00002,SearchEngine Architecture Overview,Architecture,Quinn Nakamura,The DataVault architecture follows a hexagonal...
3,DOC00003,EventBus Operations Guide,Runbooks,Casey Petrov,This runbook documents the steps to clearing a...
4,DOC00004,ModelServing ETL Runbook,Data Engineering,Morgan Chen,This guide covers the data model for Analytics...


## 3. Generate Embeddings

We embed the content of each document locally using Ollama. The embedding is stored as a string and will be cast to the native Databricks `ARRAY<FLOAT>` type at query time.

In [10]:
embeddings = []
for doc in tqdm(documents, desc = "Generating embeddings"):
    embeddings.append(get_embedding(doc["CONTENT"]))

df["EMBEDDING"] = [str(e) for e in embeddings]
print(f"Generated {len(embeddings)} embeddings.")

Generating embeddings:   0%|          | 0/200 [00:00<?, ?it/s]

Generated 200 embeddings.


## 4. Load Data into Databricks

We create a Delta table and load the documents using parameterized `INSERT` statements. The embedding is stored as a `STRING` and cast to `ARRAY<FLOAT>` at query time for similarity computation.

Delta Lake is the default table format in Databricks - it provides ACID transactions, schema enforcement and time travel out of the box.

In [11]:
cursor.execute(f"DROP TABLE IF EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}")

cursor.execute(f"""
    CREATE TABLE {CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME} (
        DOC_ID    STRING,
        TITLE     STRING,
        CATEGORY  STRING,
        AUTHOR    STRING,
        CONTENT   STRING,
        EMBEDDING STRING
    )
    USING DELTA
""")

print(f"Table '{TABLE_NAME}' created.")

Table 'documents' created.


In [12]:
values = []
for i, doc in enumerate(documents):
    embedding_str = json.dumps(embeddings[i])
    # Escape single quotes in text fields
    title    = doc["TITLE"].replace("'", "\\'")
    category = doc["CATEGORY"].replace("'", "\\'")
    author   = doc["AUTHOR"].replace("'", "\\'")
    content  = doc["CONTENT"].replace("'", "\\'")
    emb      = embedding_str.replace("'", "\\'")
    values.append(
        f"('{doc['DOC_ID']}', '{title}', '{category}', '{author}', '{content}', '{emb}')"
    )

cursor.execute(f"""
    INSERT INTO {CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}
    (DOC_ID, TITLE, CATEGORY, AUTHOR, CONTENT, EMBEDDING)
    VALUES {', '.join(values)}
""")

cursor.execute(f"SELECT COUNT(*) FROM {CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}")
count = cursor.fetchone()[0]

print(f"Loaded {count} documents into {TABLE_NAME}.")

Loaded 200 documents into documents.


## 5. Semantic Search

We embed the user's query locally with Ollama and pass it to Databricks SQL as a parameter. The query uses `VECTOR_COSINE_SIMILARITY` to rank documents by similarity to the query.

The stored embedding string is parsed using `FROM_JSON` and cast to `ARRAY<FLOAT>` before computing cosine similarity. This two-step cast is required because the Databricks SQL connector does not support the `VECTOR` type directly for inserts, so we store embeddings as `STRING` and cast at query time.

In [13]:
def search_docs(query: str, top_k: int = 5):
    query_embedding = get_embedding(query)
    query_json = json.dumps(query_embedding)

    cursor.execute(f"""
        SELECT
            DOC_ID,
            TITLE,
            CATEGORY,
            AUTHOR,
            CONTENT,
            VECTOR_COSINE_SIMILARITY(
                FROM_JSON(EMBEDDING, 'ARRAY<FLOAT>'),
                FROM_JSON(?, 'ARRAY<FLOAT>')
            ) AS similarity
        FROM {CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}
        ORDER BY similarity DESC
        LIMIT {top_k}
    """, (query_json,))

    results = cursor.fetchall()
    print(f"\nQuery: '{query}'\n")
    for r in results:
        doc_id, title, category, author, content, similarity = r
        print(f"  {doc_id} | {category} | {author}")
        print(f"  Title: {title}")
        print(f"  Similarity: {similarity:.3f}")
        print(f"  Content: {content[:150]}...")
        print()

In [14]:
search_docs("how do I restart a service during an incident")


Query: 'how do I restart a service during an incident'

  DOC00108 | Runbooks | Reese Fontaine
  Title: ReportingService Operations Guide
  Similarity: 0.553
  Content: This runbook covers the procedure for restarting the service in the EventBus environment. Follow these steps in order during an incident. First, verif...

  DOC00039 | Runbooks | Casey Petrov
  Title: Runbook: database connection exhaustion in EventBus
  Similarity: 0.508
  Content: This runbook documents the steps to restarting the service for the DataVault service. Prerequisites: access to the admin panel. Steps: (1) Verify aler...

  DOC00088 | Runbooks | Casey Petrov
  Title: DataVault Runbook: rolling back a deployment
  Similarity: 0.492
  Content: This runbook documents the steps to restarting the service for the SearchEngine service. Prerequisites: access to production console. Steps: (1) Verif...

  DOC00107 | Runbooks | Quinn Nakamura
  Title: ModelServing Operations Guide
  Similarity: 0.433
  Content: This 

In [15]:
search_docs("what are the security requirements for storing sensitive data")


Query: 'what are the security requirements for storing sensitive data'

  DOC00093 | Security | Jordan Blake
  Title: EventBus Data Classification Guide
  Similarity: 0.576
  Content: This document covers the data classification and handling requirements for DataVault. Data classified as restricted must be encrypted at rest using TL...

  DOC00114 | Security | Casey Petrov
  Title: DataPlatform Data Classification Guide
  Similarity: 0.574
  Content: This document covers the data classification and handling requirements for DataVault. Data classified as confidential must be encrypted at rest using ...

  DOC00104 | Security | Reese Fontaine
  Title: ModelServing Security Policy
  Similarity: 0.558
  Content: This document outlines the security requirements for ModelServing. All data in transit must use end-to-end encryption. Access is controlled via role-b...

  DOC00169 | Security | Skyler Mbeki
  Title: EventBus Access Control Policy
  Similarity: 0.556
  Content: This document outl

In [16]:
search_docs("how do I deploy a machine learning model to production")


Query: 'how do I deploy a machine learning model to production'

  DOC00087 | ML Platform | Reese Fontaine
  Title: AnalyticsAPI Feature Store Guide
  Similarity: 0.545
  Content: This document covers model deployment on the SearchEngine platform. Models must pass performance, latency and fairness checks before promotion to prod...

  DOC00028 | Onboarding | Taylor Osei
  Title: platform Team Starter Guide
  Similarity: 0.516
  Content: Welcome to the ML team. This guide covers everything you need to get set up in your first 30 days. You will need access to Slack, GitHub and Jira. You...

  DOC00186 | ML Platform | Casey Petrov
  Title: DataPlatform ML Platform Guide
  Similarity: 0.512
  Content: This document covers model deployment on the ReportingService platform. Models must pass unit test, integration test and bias checks before promotion ...

  DOC00136 | Onboarding | Avery Singh
  Title: security Team Starter Guide
  Similarity: 0.503
  Content: Welcome to the infrastructure t

## 6. Filtered Search

Because the documents live in a standard Delta table, filtering is plain SQL. We add `WHERE` clauses to the similarity query - any column can be used as a filter with no additional setup.

In [17]:
def search_docs_filtered(
    query:    str,
    category: str = None,
    author:   str = None,
    top_k:    int = 5
):
    query_embedding = get_embedding(query)
    query_json = json.dumps(query_embedding)

    filters = []
    if category: filters.append(f"CATEGORY = '{category}'")
    if author:   filters.append(f"AUTHOR = '{author}'")

    where_clause = "WHERE " + " AND ".join(filters) if filters else ""

    cursor.execute(f"""
        SELECT
            DOC_ID,
            TITLE,
            CATEGORY,
            AUTHOR,
            CONTENT,
            VECTOR_COSINE_SIMILARITY(
                FROM_JSON(EMBEDDING, 'ARRAY<FLOAT>'),
                FROM_JSON(?, 'ARRAY<FLOAT>')
            ) AS similarity
        FROM {CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}
        {where_clause}
        ORDER BY similarity DESC
        LIMIT {top_k}
    """, (query_json,))

    results = cursor.fetchall()
    label = f"query = '{query}'"
    if category: label += f", category = '{category}'"
    if author:   label += f", author = '{author}'"
    print(f"\n{label}\n")

    for r in results:
        doc_id, title, category_r, author_r, content, similarity = r
        print(f"  {doc_id} | {category_r} | {author_r}")
        print(f"  Title: {title}")
        print(f"  Similarity: {similarity:.3f}")
        print(f"  Content: {content[:150]}...")
        print()

In [18]:
# Runbooks only - useful during an incident
search_docs_filtered(
    "steps to roll back a failed deployment",
    category = "Runbooks"
)


query = 'steps to roll back a failed deployment', category = 'Runbooks'

  DOC00021 | Runbooks | Taylor Osei
  Title: DataPlatform Runbook: restarting the service
  Similarity: 0.575
  Content: This runbook covers the procedure for rolling back a deployment in the SearchEngine environment. Follow these steps in order during an incident. First...

  DOC00138 | Runbooks | Riley Kumar
  Title: Runbook: high error rates in DataVault
  Similarity: 0.547
  Content: This runbook covers the procedure for rolling back a deployment in the DataVault environment. Follow these steps in order during an incident. First, v...

  DOC00184 | Runbooks | Avery Singh
  Title: Runbook: latency spikes in DataPlatform
  Similarity: 0.542
  Content: This runbook documents the steps to rolling back a deployment for the SearchEngine service. Prerequisites: access to VPN access. Steps: (1) Verify ale...

  DOC00000 | Runbooks | Morgan Chen
  Title: DataVault Runbook: rolling back a deployment
  Similarity: 0.507

In [19]:
# Security documents only
search_docs_filtered(
    "password rotation and credential management policy",
    category = "Security"
)


query = 'password rotation and credential management policy', category = 'Security'

  DOC00193 | Security | Skyler Mbeki
  Title: StreamProcessor Security Policy
  Similarity: 0.414
  Content: The security policy for StreamProcessor requires MFA for all production access. Credentials must be rotated every 90 days. Multi-factor authentication...

  DOC00005 | Security | Riley Kumar
  Title: SearchEngine Security Policy
  Similarity: 0.407
  Content: The security policy for ReportingService requires encrypted storage for PII. Credentials must be rotated every 60 days. Multi-factor authentication is...

  DOC00044 | Security | Riley Kumar
  Title: SearchEngine Access Control Policy
  Similarity: 0.399
  Content: The security policy for SearchEngine requires all secrets to be stored in the vault. Credentials must be rotated every 30 days. Multi-factor authentic...

  DOC00050 | Security | Taylor Osei
  Title: ModelServing Data Classification Guide
  Similarity: 0.367
  Content: The secur

In [20]:
# Data Engineering documents only
search_docs_filtered(
    "how does the data pipeline handle failures and retries",
    category = "Data Engineering"
)


query = 'how does the data pipeline handle failures and retries', category = 'Data Engineering'

  DOC00103 | Data Engineering | Avery Singh
  Title: SearchEngine Data Pipeline Documentation
  Similarity: 0.607
  Content: The ModelServing data pipeline runs on hourly and processes 2 million records per run. It reads from the production database, applies normalisation an...

  DOC00018 | Data Engineering | Morgan Chen
  Title: ModelServing Data Pipeline Documentation
  Similarity: 0.556
  Content: The ReportingService data pipeline runs on hourly and processes 50 million records per run. It reads from the CRM system, applies deduplication and en...

  DOC00106 | Data Engineering | Avery Singh
  Title: EventBus Data Pipeline Documentation
  Similarity: 0.547
  Content: This document describes the streaming pipeline for the DataVault data product. Data is ingested from the production database every quarterly. Transfor...

  DOC00146 | Data Engineering | Morgan Chen
  Title: DataPlatform 

## 7. Analytics with SQL

Because the documents live in a standard Delta table, we can run analytical SQL queries alongside semantic search. Here we summarize document counts and authorship by category.

In [21]:
cursor.execute(f"""
    SELECT
        CATEGORY,
        COUNT(*)                    AS TOTAL_DOCS,
        COUNT(DISTINCT AUTHOR)      AS UNIQUE_AUTHORS,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS PCT_OF_TOTAL
    FROM {CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}
    GROUP BY CATEGORY
    ORDER BY TOTAL_DOCS DESC
""")

summary = cursor.fetchall()
print(f"{'CATEGORY':<20} {'DOCS':>6} {'AUTHORS':>8} {'PCT':>6}")
print("-" * 44)
for row in summary:
    print(f"  {row[0]:<18} {row[1]:>6} {row[2]:>8} {row[3]:>5}%")

CATEGORY               DOCS  AUTHORS    PCT
--------------------------------------------
  Runbooks               30       10  15.0%
  Incident Reports       30        9  15.0%
  Data Engineering       25       10  12.5%
  Onboarding             25        9  12.5%
  Architecture           24        9  12.0%
  ML Platform            24        9  12.0%
  Security               22       10  11.0%
  API                    20        9  10.0%


## 8. Lakebase - pgvector Inside the Lakehouse

Databricks Lakebase is a fully managed Postgres database integrated directly into the Databricks platform. It is the product of Databricks' acquisition of Neon in 2025 and brings serverless Postgres with `pgvector` into the lakehouse context.

This section demonstrates connecting to Lakebase using a manually obtained OAuth token - the simplest approach for a tutorial. For production use, the recommended approach is OAuth token rotation via a Databricks service principal and the `generate_database_credential()` method from the Databricks SDK.

### Prerequisites

1. Create a Lakebase project in the Databricks UI under **Lakebase Postgres**
2. Click **Connect**, select branch and database, then copy the **OAuth token** and **connection string host**
3. Set the following environment variables:

```bash
export LAKEBASE_HOST="ep-xxxx.database.xxxx.cloud.databricks.com"
export LAKEBASE_USER="your-email-address"
export LAKEBASE_TOKEN="your-oauth-token"
export LAKEBASE_DBNAME="databricks_postgres"
```

> **Note:** The OAuth token expires after 1 hour. If the connection fails, generate a fresh token from the Lakebase Connect dialog.

In [22]:
%pip install psycopg2-binary==2.9.12 --quiet

Note: you may need to restart the kernel to use updated packages.


In [23]:
import psycopg2

In [24]:
LAKEBASE_HOST   = os.environ["LAKEBASE_HOST"]
LAKEBASE_USER   = os.environ["LAKEBASE_USER"]
LAKEBASE_TOKEN  = os.environ["LAKEBASE_TOKEN"]
LAKEBASE_DBNAME = os.environ["LAKEBASE_DBNAME"]
LAKEBASE_TABLE  = "wiki_documents"

In [25]:
lb_conn      = psycopg2.connect(
    host     = LAKEBASE_HOST,
    user     = LAKEBASE_USER,
    password = LAKEBASE_TOKEN,
    dbname   = LAKEBASE_DBNAME,
    sslmode  = "require",
    port     = 5432
)

lb_conn.autocommit = True
lb_cursor = lb_conn.cursor()

print("Connected to Lakebase.")

Connected to Lakebase.


In [26]:
# Enable pgvector extension
lb_cursor.execute("CREATE EXTENSION IF NOT EXISTS vector;")

print("pgvector extension enabled.")

pgvector extension enabled.


In [27]:
# Verify pgvector is enabled
lb_cursor.execute("SELECT extversion FROM pg_extension WHERE extname = 'vector';")
result = lb_cursor.fetchone()
assert result is not None, "pgvector is not installed. Run CREATE EXTENSION vector;"

print(f"pgvector version: {result[0]}")

pgvector version: 0.8.0


In [28]:
# Create table with native vector column
lb_cursor.execute(f"DROP TABLE IF EXISTS {LAKEBASE_TABLE}")
lb_cursor.execute(f"""
    CREATE TABLE {LAKEBASE_TABLE} (
        doc_id    TEXT PRIMARY KEY,
        title     TEXT,
        category  TEXT,
        author    TEXT,
        content   TEXT,
        embedding vector({EMBEDDING_DIMS})
    )
""")

print(f"Table '{LAKEBASE_TABLE}' created with vector({EMBEDDING_DIMS}) column.")

Table 'wiki_documents' created with vector(384) column.


In [29]:
# Insert documents reusing embeddings already generated
print(f"Inserting {len(documents)} documents...")

for i, doc in enumerate(documents):
    lb_cursor.execute(f"""
        INSERT INTO {LAKEBASE_TABLE} (doc_id, title, category, author, content, embedding)
        VALUES (%s, %s, %s, %s, %s, %s)
    """, (
        doc["DOC_ID"],
        doc["TITLE"],
        doc["CATEGORY"],
        doc["AUTHOR"],
        doc["CONTENT"],
        str(embeddings[i])
    ))

lb_cursor.execute(f"SELECT COUNT(*) FROM {LAKEBASE_TABLE}")
count = lb_cursor.fetchone()[0]

print(f"Loaded {count} documents into Lakebase.")

Inserting 200 documents...
Loaded 200 documents into Lakebase.


In [30]:
# Create HNSW index for fast similarity search
lb_cursor.execute(f"""
    CREATE INDEX ON {LAKEBASE_TABLE}
    USING hnsw (embedding vector_cosine_ops)
""")
print("HNSW index created.")

HNSW index created.


In [31]:
def search_lakebase(query: str, top_k: int = 5):
    query_embedding = get_embedding(query)
    lb_cursor.execute(f"""
        SELECT
            doc_id,
            title,
            category,
            author,
            content,
            1 - (embedding <=> %s::vector) AS similarity
        FROM {LAKEBASE_TABLE}
        ORDER BY embedding <=> %s::vector
        LIMIT %s
    """, (str(query_embedding), str(query_embedding), top_k))

    results = lb_cursor.fetchall()
    print(f"\nLakebase query: '{query}'\n")
    for r in results:
        doc_id, title, category, author, content, similarity = r
        print(f"  {doc_id} | {category} | {author}")
        print(f"  Title: {title}")
        print(f"  Similarity: {similarity:.3f}")
        print(f"  Content: {content[:150]}...")
        print()

In [32]:
search_lakebase("how do I restart a service during an incident")


Lakebase query: 'how do I restart a service during an incident'

  DOC00108 | Runbooks | Reese Fontaine
  Title: ReportingService Operations Guide
  Similarity: 0.553
  Content: This runbook covers the procedure for restarting the service in the EventBus environment. Follow these steps in order during an incident. First, verif...

  DOC00039 | Runbooks | Casey Petrov
  Title: Runbook: database connection exhaustion in EventBus
  Similarity: 0.508
  Content: This runbook documents the steps to restarting the service for the DataVault service. Prerequisites: access to the admin panel. Steps: (1) Verify aler...

  DOC00088 | Runbooks | Casey Petrov
  Title: DataVault Runbook: rolling back a deployment
  Similarity: 0.492
  Content: This runbook documents the steps to restarting the service for the SearchEngine service. Prerequisites: access to production console. Steps: (1) Verif...

  DOC00107 | Runbooks | Quinn Nakamura
  Title: ModelServing Operations Guide
  Similarity: 0.433
  Conte

In [33]:
lb_cursor.execute(f"DROP TABLE IF EXISTS {LAKEBASE_TABLE}")
print(f"Table '{LAKEBASE_TABLE}' dropped.")

lb_cursor.close()
lb_conn.close()
print("Lakebase connection closed.")

Table 'wiki_documents' dropped.
Lakebase connection closed.


## Cleanup

In [34]:
cursor.execute(f"DROP TABLE IF EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}")
print(f"Table '{TABLE_NAME}' dropped.")

cursor.execute(f"DROP SCHEMA IF EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")
print(f"Schema '{SCHEMA_NAME}' dropped.")

cursor.close()
conn.close()
print("Connection closed.")

Table 'documents' dropped.
Schema 'wiki_docs' dropped.
Connection closed.
